# LeetCode 3563 — Lexicographically Smallest String After Adjacent Removals bằng Python

- Link: https://leetcode.com/problems/lexicographically-smallest-string-after-adjacent-removals/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động khoảng** (interval DP: đoạn nào xóa sạch được), **DP xây đáp án theo thứ tự từ điển**, mô hình "ký tự giữ lại + khe bị xóa"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lexicographicallySmallestString(self, s: str) -> str:
```

## 1. Mục tiêu học

1. Hiểu phép xóa "cặp ký tự kề nhau liên tiếp trong bảng chữ cái (vòng tròn `a`–`z`)" và cấu trúc **lồng nhau như dấu ngoặc** của các cặp bị xóa.
2. Viết **interval DP** `rem[i][j]`: đoạn `s[i:j]` có xóa sạch được không, bằng cách ghép `s[i]` với một `s[k]` nào đó.
3. Mô hình hóa chuỗi kết quả = các ký tự **giữ lại**, giữa chúng là các **khe xóa sạch được** ⇒ DP `ans[i]` = chuỗi nhỏ nhất thu được từ hậu tố `s[i:]`.
4. Nắm quy tắc so sánh thứ tự từ điển trong Python (`""` nhỏ nhất, tiền tố nhỏ hơn chuỗi dài hơn).
5. Kiểm chứng bằng **duyệt mọi chuỗi đạt được** (DFS trên tập trạng thái) với chuỗi ngắn.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chữ thường. Có thể làm thao tác sau **bao nhiêu lần tùy ý** (kể cả 0):

- Xóa một **cặp ký tự kề nhau** mà hai ký tự **liên tiếp trong bảng chữ cái** (theo thứ tự nào cũng được: `"ab"` hoặc `"ba"`), rồi dồn chuỗi lại.
- Bảng chữ cái là **vòng tròn**: `'a'` và `'z'` cũng liên tiếp.

Trả về chuỗi **nhỏ nhất theo thứ tự từ điển** có thể thu được.

**Ví dụ 1:** `"abc"` → `"a"` (xóa `"bc"`).

**Ví dụ 2:** `"bcda"` → `""` (xóa `"cd"` còn `"ba"`, xóa tiếp).

**Ví dụ 3:** `"zdce"` → `"zdce"` (xóa `"dc"` được `"ze"`, nhưng `"zdce" < "ze"`, nên không xóa là tốt nhất).

**Ràng buộc:** `1 ≤ len(s) ≤ 250`.

In [1]:
examples = [("abc", "a"), ("bcda", ""), ("zdce", "zdce")]
for s, exp in examples:
    print(repr(s), '-> kỳ vọng', repr(exp))

'abc' -> kỳ vọng 'a'
'bcda' -> kỳ vọng ''
'zdce' -> kỳ vọng 'zdce'


## 3. Kiến thức nền

### 3.1. Hai ký tự "liên tiếp" trên vòng chữ cái

`|ord(a) − ord(b)| = 1`, hoặc `= 25` (cặp `a`–`z`).

In [2]:
def is_adj(a, b):
    d = abs(ord(a) - ord(b))
    return d == 1 or d == 25

print(is_adj('a', 'b'), is_adj('b', 'a'), is_adj('a', 'z'), is_adj('a', 'c'), is_adj('m', 'm'))

True True True False False


### 3.2. So sánh thứ tự từ điển trong Python

Python so sánh chuỗi từng ký tự; nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn. Chuỗi rỗng `""` nhỏ hơn mọi chuỗi khác. `min()` trên danh sách chuỗi dùng đúng quy tắc này.

In [3]:
print("zdce" < "ze", "a" < "ab", "" < "a", min(["ba", "b", "abz"]))

True True True abz


### 3.3. Cấu trúc "dấu ngoặc" của các phép xóa

Mỗi lần xóa ghép hai ký tự **đang kề nhau**. Nếu ghép `s[i]` với `s[k]` (ban đầu không kề), mọi ký tự giữa chúng phải đã bị xóa trước đó — và chúng chỉ có thể ghép **với nhau** (không vượt qua `s[i]` hay `s[k]`, vì hai ký tự này còn tồn tại). Vậy các cặp bị xóa **lồng nhau hoặc rời nhau**, đúng như các cặp ngoặc trong một biểu thức ngoặc hợp lệ.

### 3.4. Interval DP

Mẫu: `ok[i][j]` cho đoạn nửa mở `s[i:j]`, tính theo độ dài tăng dần. Với đoạn xóa sạch: `s[i]` phải ghép với một `s[k]` (`i < k < j`), phần trong `s[i+1:k]` và phần sau `s[k+1:j]` đều xóa sạch được.

In [4]:
def removable_table(s):
    n = len(s)
    rem = [[False] * (n + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        rem[i][i] = True                     # đoạn rỗng luôn "xóa sạch"
    for L in range(2, n + 1, 2):             # chỉ độ dài chẵn mới có thể xóa sạch
        for i in range(n - L + 1):
            j = i + L
            for k in range(i + 1, j, 2):     # s[i+1:k] phải có độ dài chẵn
                if is_adj(s[i], s[k]) and rem[i + 1][k] and rem[k + 1][j]:
                    rem[i][j] = True
                    break
    return rem

rem = removable_table("bcda")
print('xóa sạch được: ', [("bcda"[i:j]) for i in range(5) for j in range(i + 1, 5) if rem[i][j]])

xóa sạch được:  ['bc', 'bcda', 'cd']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chuỗi cuối cùng gồm các ký tự **không bị xóa**, giữ nguyên thứ tự. Giữa hai ký tự giữ lại liên tiếp (và trước ký tự đầu, sau ký tự cuối) là một **khe** các ký tự đã bị xóa — và vì ký tự giữ lại "chặn" mọi phép ghép đi qua nó, mỗi khe phải **tự xóa sạch** được.

Vậy: chọn ký tự giữ lại đầu tiên `s[j]` sao cho `s[i:j]` xóa sạch được, rồi giải tiếp bài toán con trên `s[j+1:]`; hoặc xóa sạch toàn bộ `s[i:]`.

Tính tay `"abc"`:

- `rem("ab") = True`, `rem("bc") = True`, `rem("abc") = False` (độ dài lẻ).
- `ans(2) = min("c")`, có `rem("c")`? Không (lẻ) ⇒ `"c"`.
- `ans(1)`: xóa sạch `"bc"` ⇒ `""` ⇒ `ans(1) = ""`.
- `ans(0)`: giữ `a` + `ans(1) = "a"`; khe `"ab"` rồi giữ `c`: `"c"`; xóa sạch `"abc"`: không ⇒ `min("a", "c") = "a"`.

In [5]:
s = "abc"
rem = removable_table(s)
for i in range(len(s)):
    for j in range(i, len(s)):
        if rem[i][j]:
            print(f"từ vị trí {i}: khe {s[i:j]!r} xóa sạch được -> có thể giữ {s[j]!r} làm ký tự đầu")

từ vị trí 0: khe '' xóa sạch được -> có thể giữ 'a' làm ký tự đầu
từ vị trí 0: khe 'ab' xóa sạch được -> có thể giữ 'c' làm ký tự đầu
từ vị trí 1: khe '' xóa sạch được -> có thể giữ 'b' làm ký tự đầu
từ vị trí 2: khe '' xóa sạch được -> có thể giữ 'c' làm ký tự đầu


## 5. Cách 1 — Brute Force: duyệt mọi chuỗi đạt được

DFS từ `s`: ở mỗi chuỗi, thử xóa mọi cặp kề hợp lệ, lưu các chuỗi đã thấy. Kết quả là `min` của tập.

In [6]:
def smallest_brute(s):
    seen = {s}
    stack = [s]
    while stack:
        x = stack.pop()
        for i in range(len(x) - 1):
            if is_adj(x[i], x[i + 1]):
                y = x[:i] + x[i + 2:]
                if y not in seen:
                    seen.add(y)
                    stack.append(y)
    return min(seen)

for s, exp in examples:
    print(repr(smallest_brute(s)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'' | kỳ vọng ''
'zdce' | kỳ vọng 'zdce'


**Độ phức tạp:** số chuỗi đạt được có thể **mũ** theo `n` (mỗi chuỗi con của `s` thu được bằng cách xóa một tập các cặp lồng nhau). Chỉ dùng với `n ≤ 12`.

**Nút thắt:** ta lưu cả chuỗi trung gian. Theo mục 4, chỉ cần biết **đoạn nào xóa sạch được** và chọn ký tự giữ lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề cấu trúc.** Một chuỗi `t` đạt được từ `s` ⇔ `t` là dãy con `s[j1] s[j2] … s[jm]` (`j1 < … < jm`) sao cho các khe `s[0:j1]`, `s[j1+1:j2]`, …, `s[jm+1:n]` đều **xóa sạch được** (riêng từng khe).

**DP 1 — `rem[i][j]`:** `s[i:j]` xóa sạch ⇔ độ dài chẵn và tồn tại `k` với `is_adj(s[i], s[k])`, `rem[i+1][k]`, `rem[k+1][j]`.

**DP 2 — `ans[i]`:** chuỗi nhỏ nhất thu được từ `s[i:]`:

`ans[i] = min( "" nếu rem[i][n],  s[j] + ans[j+1] với mọi j ≥ i có rem[i][j] )`.

Vì sao chỉ cần `ans[j+1]` tối ưu cho phần sau? Với cùng ký tự đầu `s[j]`, chuỗi `s[j] + X` nhỏ nhất khi `X` nhỏ nhất — thứ tự từ điển "tôn trọng" tiền tố chung.

Quy tắc:

- **Nếu** hai ký tự liên tiếp trên vòng chữ cái và phần giữa xóa sạch **thì** cả cặp xóa được.
- **Nếu** khe trước `s[j]` xóa sạch **thì** `s[j]` có thể là ký tự đầu của kết quả.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ (top-down)

Viết trực tiếp hai định nghĩa bằng `lru_cache`. Dễ đọc, nhưng độ sâu đệ quy có thể tới `n` (chú ý giới hạn đệ quy) và chi phí gọi hàm cao.

In [7]:
import sys
from functools import lru_cache
sys.setrecursionlimit(10000)

def smallest_memo(s):
    n = len(s)

    @lru_cache(maxsize=None)
    def can(i, j):
        if i == j:
            return True
        if (j - i) % 2:
            return False
        return any(is_adj(s[i], s[k]) and can(i + 1, k) and can(k + 1, j)
                   for k in range(i + 1, j, 2))

    @lru_cache(maxsize=None)
    def best(i):
        cands = [s[j] + best(j + 1) for j in range(i, n) if can(i, j)]
        if can(i, n):
            cands.append("")
        return min(cands)

    return best(0)

for s, exp in examples:
    print(repr(smallest_memo(s)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'' | kỳ vọng ''
'zdce' | kỳ vọng 'zdce'


Lưu ý: `best(n)` khi `i = n`: `can(n, n)` đúng ⇒ trả `""` — hợp lệ.

### 7.2. Cách 3 — Bottom-up (lời giải chính)

Tính `rem` theo độ dài tăng dần, rồi `ans` từ phải sang trái.

In [8]:
def smallest_bottom_up(s):
    n = len(s)
    rem = removable_table(s)
    ans = [""] * (n + 1)
    for i in range(n - 1, -1, -1):
        best = "" if rem[i][n] else None
        for j in range(i, n):
            if rem[i][j]:
                c = s[j] + ans[j + 1]
                if best is None or c < best:
                    best = c
        ans[i] = best
    return ans[0]

for s, exp in examples:
    print(repr(smallest_bottom_up(s)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'' | kỳ vọng ''
'zdce' | kỳ vọng 'zdce'


`best` không bao giờ là `None` sau vòng lặp vì `rem[i][i]` luôn đúng (giữ `s[i]` làm ký tự đầu).

**Độ phức tạp:** `rem`: `O(n^3/8)` (chỉ độ dài chẵn, `k` bước 2); `ans`: `O(n^2)` phép so sánh chuỗi, mỗi phép `O(n)` ⇒ `O(n^3)`. Với `n = 250` chạy cỡ 0,1 s.

### 7.3. Cách 4 — Tối ưu so sánh: chọn ký tự đầu nhỏ nhất trước

Một cải tiến tự nhiên: trong các `j` hợp lệ, chỉ những `j` có `s[j]` **nhỏ nhất** mới có thể thắng (trừ phương án `""`). Lọc trước theo ký tự rồi mới so sánh phần đuôi ⇒ giảm số phép so sánh chuỗi dài.

In [9]:
def smallest_filtered(s):
    n = len(s)
    rem = removable_table(s)
    ans = [""] * (n + 1)
    for i in range(n - 1, -1, -1):
        if rem[i][n]:
            ans[i] = ""
            continue
        js = [j for j in range(i, n) if rem[i][j]]
        c0 = min(s[j] for j in js)
        ans[i] = min(s[j] + ans[j + 1] for j in js if s[j] == c0)
    return ans[0]

for s, exp in examples:
    print(repr(smallest_filtered(s)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'' | kỳ vọng ''
'zdce' | kỳ vọng 'zdce'


Nếu `rem[i][n]` đúng thì `""` thắng mọi thứ, nên có thể `continue` ngay.

## 8. TRACE từng bước

Trace cách 3 trên `s = "zdce"` (ví dụ 3) và `s = "cbadz"`: in bảng `rem` (các đoạn xóa sạch được) và `ans[i]` từ phải sang trái kèm các ứng viên.

In [10]:
def trace(s):
    n = len(s)
    rem = removable_table(s)
    print(f"s = {s!r}")
    print('  đoạn xóa sạch được (không rỗng):', [(i, j, s[i:j]) for i in range(n) for j in range(i + 2, n + 1) if rem[i][j]])
    ans = [""] * (n + 1)
    for i in range(n - 1, -1, -1):
        cands = []
        if rem[i][n]:
            cands.append(('xóa sạch s[%d:]' % i, ""))
        for j in range(i, n):
            if rem[i][j]:
                cands.append((f"giữ s[{j}]={s[j]!r} (khe {s[i:j]!r})", s[j] + ans[j + 1]))
        ans[i] = min(c for _, c in cands)
        print(f"  ans[{i}] = {ans[i]!r:8} | ứng viên: " + ', '.join(f"{d}->{c!r}" for d, c in cands))
    return ans[0]

trace("zdce")
print()
trace("cbadz")

s = 'zdce'
  đoạn xóa sạch được (không rỗng): [(1, 3, 'dc')]
  ans[3] = 'e'      | ứng viên: giữ s[3]='e' (khe '')->'e'
  ans[2] = 'ce'     | ứng viên: giữ s[2]='c' (khe '')->'ce'
  ans[1] = 'dce'    | ứng viên: giữ s[1]='d' (khe '')->'dce', giữ s[3]='e' (khe 'dc')->'e'
  ans[0] = 'zdce'   | ứng viên: giữ s[0]='z' (khe '')->'zdce'

s = 'cbadz'
  đoạn xóa sạch được (không rỗng): [(0, 2, 'cb'), (0, 4, 'cbad'), (1, 3, 'ba')]
  ans[4] = 'z'      | ứng viên: giữ s[4]='z' (khe '')->'z'
  ans[3] = 'dz'     | ứng viên: giữ s[3]='d' (khe '')->'dz'
  ans[2] = 'adz'    | ứng viên: giữ s[2]='a' (khe '')->'adz'
  ans[1] = 'badz'   | ứng viên: giữ s[1]='b' (khe '')->'badz', giữ s[3]='d' (khe 'ba')->'dz'
  ans[0] = 'adz'    | ứng viên: giữ s[0]='c' (khe '')->'cbadz', giữ s[2]='a' (khe 'cb')->'adz', giữ s[4]='z' (khe 'cbad')->'z'


'adz'

Với `"cbadz"`: `"cb"`, `"ba"`, `"az"` (vòng tròn) đều xóa được, và `"cbad"` cũng xóa sạch được (`c` ghép `d`, bên trong là `"ba"`). Xóa `"cbad"` còn `"z"`, nhưng giữ `"adz"` (chỉ xóa `"cb"`) cho chuỗi nhỏ hơn vì `'a' < 'z'` ⇒ đáp án `"adz"`.

In [11]:
print('vét cạn:', repr(smallest_brute("cbadz")), '| DP:', repr(smallest_bottom_up("cbadz")))

vét cạn: 'adz' | DP: 'adz'


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def lexicographicallySmallestString(self, s: str) -> str:
        n = len(s)
        o = [ord(c) for c in s]

        def adj(a, b):
            d = abs(a - b)
            return d == 1 or d == 25

        rem = [[False] * (n + 1) for _ in range(n + 1)]
        for i in range(n + 1):
            rem[i][i] = True
        for L in range(2, n + 1, 2):
            for i in range(n - L + 1):
                j = i + L
                ri = rem[i + 1]
                for k in range(i + 1, j, 2):
                    if adj(o[i], o[k]) and ri[k] and rem[k + 1][j]:
                        rem[i][j] = True
                        break

        ans = [""] * (n + 1)
        for i in range(n - 1, -1, -1):
            if rem[i][n]:
                ans[i] = ""
                continue
            best = None
            row = rem[i]
            for j in range(i, n):
                if row[j]:
                    c = s[j] + ans[j + 1]
                    if best is None or c < best:
                        best = c
            ans[i] = best
        return ans[0]

for s, exp in examples:
    got = Solution().lexicographicallySmallestString(s)
    print(repr(got), 'OK' if got == exp else 'SAI')

'a' OK
'' OK
'zdce' OK


## 10. Vì sao thuật toán đúng

1. **Cặp bị xóa lồng nhau (mục 3.3)** ⇒ một đoạn xóa sạch được ⇔ ký tự đầu `s[i]` ghép với một `s[k]` liên tiếp chữ cái, phần trong `s[i+1:k]` xóa sạch (trước khi `s[i]`, `s[k]` kề nhau) và phần còn lại `s[k+1:j]` xóa sạch (độc lập). Chiều ngược lại: có các phép xóa như vậy thì thực hiện phần trong, rồi cặp `(i, k)`, rồi phần sau.
2. **Bổ đề cấu trúc:** ký tự giữ lại không bao giờ bị ghép, nên không cặp nào "vượt qua" nó ⇒ mỗi khe tự xóa sạch. Ngược lại, nếu mọi khe xóa sạch được thì xóa lần lượt từng khe sẽ thu được đúng dãy con các ký tự giữ lại.
3. **DP `ans` đúng (tối ưu cấu trúc con):** mọi chuỗi đạt được từ `s[i:]` hoặc rỗng (khi `rem[i][n]`), hoặc có dạng `s[j] + t` với `rem[i][j]` và `t` đạt được từ `s[j+1:]`. Với `j` cố định, `s[j] + t` nhỏ nhất khi `t` nhỏ nhất (cùng tiền tố `s[j]`). Lấy min trên mọi phương án.
4. **Rỗng là nhỏ nhất** nên khi `rem[i][n]` đúng có thể trả về `""` ngay.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 250`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt mọi chuỗi đạt được | mũ | mũ |
| 2. Đệ quy có nhớ | `O(n^3)` + chi phí gọi hàm | `O(n^2)` + stack `O(n)` |
| 3. Bottom-up | `O(n^3)` (`rem` ≈ `n^3/48` bước) + `O(n^3)` so sánh chuỗi | `O(n^2)` |
| 4. Lọc ký tự đầu | như cách 3, ít so sánh chuỗi hơn | `O(n^2)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {
    'memo': smallest_memo,
    'bottom_up': smallest_bottom_up,
    'filtered': smallest_filtered,
    'submit': lambda s: Solution().lexicographicallySmallestString(s),
}
edge = [
    ("a", "a"),
    ("az", ""),          # vòng tròn: a và z liên tiếp
    ("za", ""),
    ("aa", "aa"),        # ký tự giống nhau không xóa được
    ("ac", "ac"),
    ("abab", ""),
    ("ba", ""),
    ("cab", "c"),        # xóa "ab" -> "c"; không thể xóa c
]
for s, exp in examples + edge:
    for name, f in impls.items():
        got = f(s)
        assert got == exp, (name, s, got, exp)
    assert smallest_brute(s) == exp
print("'bac' ->", repr(Solution().lexicographicallySmallestString("bac")), '| vét cạn:', repr(smallest_brute("bac")))
print('Ví dụ đề và ca biên: tất cả đúng.')

'bac' -> 'bac' | vét cạn: 'bac'
Ví dụ đề và ca biên: tất cả đúng.


Với `"bac"`: các chuỗi đạt được là `"bac"` và `"c"` (xóa `"ba"`); `"bac" < "c"` vì `'b' < 'c'` ⇒ đáp án là `"bac"` — một ví dụ nữa cho thấy **xóa không phải lúc nào cũng tốt**.

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [14]:
import random
random.seed(3563)
cnt = 0
for _ in range(1500):
    s = ''.join(random.choice('abcyz') for _ in range(random.randint(1, 11)))
    exp = smallest_brute(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'chuỗi ngẫu nhiên.')

Khớp vét cạn trên 1500 chuỗi ngẫu nhiên.


### 12.3. Ca lớn (`n = 250`)

In [15]:
import time
random.seed(250)
cases = {
    'ngẫu nhiên abc': ''.join(random.choice('abc') for _ in range(250)),
    'ngẫu nhiên 26 chữ': ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(250)),
    '"ab" lặp': 'ab' * 125,
    'chuỗi tăng vòng': ''.join(chr(97 + i % 26) for i in range(250)),
}
for name, s in cases.items():
    t = time.time(); a = Solution().lexicographicallySmallestString(s); t1 = time.time() - t
    t = time.time(); b = smallest_filtered(s); t2 = time.time() - t
    assert a == b
    print(f"{name:>18}: độ dài kết quả {len(a):>3} | chính {t1:.2f}s | lọc {t2:.2f}s | đầu kết quả {a[:12]!r}")

    ngẫu nhiên abc: độ dài kết quả 116 | chính 0.07s | lọc 0.09s | đầu kết quả 'bbaccacabbcc'


 ngẫu nhiên 26 chữ: độ dài kết quả 228 | chính 0.10s | lọc 0.13s | đầu kết quả 'phlxhvcclpgz'
          "ab" lặp: độ dài kết quả   0 | chính 0.01s | lọc 0.01s | đầu kết quả ''
   chuỗi tăng vòng: độ dài kết quả   0 | chính 0.01s | lọc 0.01s | đầu kết quả ''


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: cứ thấy cặp xóa được là xóa (dùng stack)

In [16]:
def greedy_stack(s):
    st = []
    for ch in s:
        if st and is_adj(st[-1], ch):
            st.pop()                    # SAI: luôn xóa khi có thể
        else:
            st.append(ch)
    return ''.join(st)

print('sai :', repr(greedy_stack("zdce")))     # "ze"
print('đúng:', repr(Solution().lexicographicallySmallestString("zdce")))
print('sai :', repr(greedy_stack("abc")), '| đúng:', repr(Solution().lexicographicallySmallestString("abc")))

sai : 'ze'
đúng: 'zdce'
sai : 'c' | đúng: 'a'


**Vì sao sai:** (1) xóa có thể làm chuỗi **lớn hơn** (`"zdce" → "ze"`); (2) chọn cặp nào để xóa quan trọng (`"abc"`: xóa `"ab"` còn `"c"`, xóa `"bc"` còn `"a"`).

### Lỗi 2 — Quên cặp vòng tròn `a`–`z`

In [17]:
def is_adj_wrong(a, b):
    return abs(ord(a) - ord(b)) == 1            # SAI: thiếu a-z

def smallest_wrong_adj(s):
    seen, st = {s}, [s]
    while st:
        x = st.pop()
        for i in range(len(x) - 1):
            if is_adj_wrong(x[i], x[i + 1]):
                y = x[:i] + x[i + 2:]
                if y not in seen:
                    seen.add(y); st.append(y)
    return min(seen)

print('sai :', repr(smallest_wrong_adj("baz")))      # 'z'? hay 'baz'
print('đúng:', repr(Solution().lexicographicallySmallestString("baz")))

sai : 'baz'
đúng: 'b'


**Vì sao sai:** đề nói rõ bảng chữ cái là vòng tròn; `"az"` xóa được ⇒ `"baz" → "b"` nhỏ hơn.

### Lỗi 3 — `rem` chỉ xét ghép **kề nhau** (`s[i]` với `s[i+1]`)

In [18]:
def rem_wrong(s):
    n = len(s)
    rem = [[False] * (n + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        rem[i][i] = True
    for L in range(2, n + 1, 2):
        for i in range(n - L + 1):
            j = i + L
            rem[i][j] = is_adj(s[i], s[i + 1]) and rem[i + 2][j]   # SAI: chỉ ghép với ký tự ngay sau
    return rem

s = "bcda"
print('sai : xóa sạch "bcda"?', rem_wrong(s)[0][4])
print('đúng: xóa sạch "bcda"?', removable_table(s)[0][4])

sai : xóa sạch "bcda"? False
đúng: xóa sạch "bcda"? True


**Vì sao sai:** `b` ghép với `a` ở cuối **sau khi** `"cd"` ở giữa đã bị xóa — cấu trúc lồng nhau như ngoặc `( () )`.

### Lỗi 4 — Chọn "ký tự đầu nhỏ nhất" nhưng lấy `j` **đầu tiên** đạt ký tự đó mà không so phần đuôi

In [19]:
def first_min_char(s):
    n = len(s)
    rem = removable_table(s)
    ans = [""] * (n + 1)
    for i in range(n - 1, -1, -1):
        if rem[i][n]:
            ans[i] = ""
            continue
        js = [j for j in range(i, n) if rem[i][j]]
        c0 = min(s[j] for j in js)
        j = next(j for j in js if s[j] == c0)        # SAI: không so phần đuôi
        ans[i] = s[j] + ans[j + 1]
    return ans[0]

s = "bacbaddazca"          # tìm được bằng cách so với lời giải đúng trên nhiều chuỗi ngẫu nhiên
found = (s, first_min_char(s), smallest_brute(s))
print('Phản ví dụ (s, sai, đúng):', found)

Phản ví dụ (s, sai, đúng): ('bacbaddazca', 'ada', 'a')


**Vì sao sai:** nhiều vị trí `j` có cùng ký tự nhỏ nhất nhưng phần đuôi `ans[j+1]` khác nhau; phải so cả chuỗi `s[j] + ans[j+1]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xóa cặp kề nhau thỏa điều kiện, lặp lại" ⇒ cấu trúc **dấu ngoặc lồng nhau** ⇒ interval DP "ghép `s[i]` với `s[k]`".
- "Chuỗi nhỏ nhất theo thứ tự từ điển sau các thao tác" ⇒ DP trên hậu tố, `ans[i] = min(lựa chọn ký tự đầu + ans[sau])`.
- Tham lam bằng stack chỉ đúng khi xóa **luôn có lợi** và không có xung đột lựa chọn — ở đây không phải.
- Độ dài lẻ không bao giờ xóa sạch ⇒ bỏ qua để giảm 1/2 – 1/4 khối lượng.

## 15. Biến thể của bài

1. Hỏi **độ dài ngắn nhất** có thể ⇒ DP `minlen[i] = min(minlen[j+1] + 1 với rem[i][j], 0 nếu rem[i][n])`.
2. Hỏi có xóa sạch được cả chuỗi không ⇒ chỉ cần `rem[0][n]`.
3. Điều kiện xóa khác (hai ký tự **giống nhau**, LC 1047) ⇒ khi đó tham lam stack đúng vì kết quả **duy nhất**.
4. Bài liên quan: LC 1246 *Palindrome Removal*, LC 664 *Strange Printer* (interval DP ghép `s[i]` với `s[k]`).

In [20]:
# Biến thể 1: độ dài ngắn nhất
def shortest_len(s):
    n = len(s)
    rem = removable_table(s)
    best = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        best[i] = 0 if rem[i][n] else min(best[j + 1] + 1 for j in range(i, n) if rem[i][j])
    return best[0]

for s in ["abc", "bcda", "zdce", "aaa"]:
    print(repr(s), '-> độ dài ngắn nhất', shortest_len(s))

'abc' -> độ dài ngắn nhất 1
'bcda' -> độ dài ngắn nhất 0
'zdce' -> độ dài ngắn nhất 2
'aaa' -> độ dài ngắn nhất 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Duyệt mọi chuỗi đạt được | mũ | mũ | Oracle `n ≤ 12` |
| 2 | Đệ quy có nhớ | `O(n^3)` | `O(n^2)` | Viết nhanh, dễ hiểu |
| 3 | Bottom-up `rem` + `ans` | `O(n^3)` | `O(n^2)` | **Nộp bài** |
| 4 | Lọc theo ký tự đầu | `O(n^3)` (ít so sánh) | `O(n^2)` | Tối ưu hằng số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"dcba"` → ?

**Bài 2 (tính tay):** `"bza"` → ?

**Bài 3 (điền TODO):** viết `can_clear(s)` trả về `True` nếu xóa sạch được cả chuỗi.

**Bài 4 (tự viết lại):** biến thể 1 — độ dài ngắn nhất, kiểm chứng bằng vét cạn.

**Bài 5 (mini challenge):** in ra **một dãy thao tác** (vị trí cặp bị xóa theo thứ tự) để đạt chuỗi tối ưu, rồi mô phỏng để kiểm tra.

In [21]:
def can_clear_todo(s):
    # TODO: removable_table(s)[0][len(s)]
    raise NotImplementedError

def shortest_len_todo(s):
    # TODO
    raise NotImplementedError

def operations_todo(s):
    # TODO: truy vết rem (lưu k) và ans (lưu j), sinh thứ tự xóa
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"dc"` và `"ba"` đều xóa được. (2) Xóa cả hai ⇒ `""`. (3) Đáp án `""`.

**Bài 2.** (1) `"bz"`? Không liên tiếp. `"za"` liên tiếp (vòng). (2) Xóa `"za"` còn `"b"`; giữ nguyên `"bza"`. (3) `"b" < "bza"` ⇒ `"b"`.

**Bài 3.** (1) Độ dài phải chẵn. (2) Dùng bảng `rem`. (3) So với vét cạn: `"" in tập đạt được`.

**Bài 4.** (1) Như mục 15. (2) Vét cạn: `min(len(t))` trên tập đạt được. (3) So trên chuỗi ngẫu nhiên.

**Bài 5.** (1) Lưu `pair_k[i][j]` khi `rem[i][j]` đúng. (2) Xóa phần trong trước, rồi cặp `(i, k)`, rồi phần sau. (3) Vị trí hiện tại = vị trí gốc trừ số ký tự đã xóa phía trước.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
a1 = Solution().lexicographicallySmallestString("dcba")
a2 = Solution().lexicographicallySmallestString("bza")
assert a1 == "" == smallest_brute("dcba")
assert a2 == "b" == smallest_brute("bza")

def can_clear(s):
    return removable_table(s)[0][len(s)]

random.seed(3)
for _ in range(500):
    s = ''.join(random.choice('abcz') for _ in range(random.randint(1, 10)))
    seen, st = {s}, [s]
    while st:
        x = st.pop()
        for i in range(len(x) - 1):
            if is_adj(x[i], x[i + 1]) and x[:i] + x[i + 2:] not in seen:
                seen.add(x[:i] + x[i + 2:]); st.append(x[:i] + x[i + 2:])
    assert can_clear(s) == ("" in seen)
print('Bài 1:', repr(a1), '| Bài 2:', repr(a2), '| Bài 3: can_clear đúng')

Bài 1: '' | Bài 2: 'b' | Bài 3: can_clear đúng


In [23]:
# Bài 4
def shortest_brute(s):
    seen, st = {s}, [s]
    while st:
        x = st.pop()
        for i in range(len(x) - 1):
            if is_adj(x[i], x[i + 1]):
                y = x[:i] + x[i + 2:]
                if y not in seen:
                    seen.add(y); st.append(y)
    return min(len(t) for t in seen)

random.seed(4)
for _ in range(500):
    s = ''.join(random.choice('abcz') for _ in range(random.randint(1, 10)))
    assert shortest_len(s) == shortest_brute(s)
print('Bài 4: shortest_len khớp vét cạn')

Bài 4: shortest_len khớp vét cạn


In [24]:
# Bài 5: sinh dãy thao tác
def operations(s):
    n = len(s)
    rem = removable_table(s)
    pk = {}
    for i in range(n):
        for j in range(i + 2, n + 1, 2):
            if rem[i][j]:
                for k in range(i + 1, j, 2):
                    if is_adj(s[i], s[k]) and rem[i + 1][k] and rem[k + 1][j]:
                        pk[(i, j)] = k
                        break
    pairs = []                               # cặp chỉ số gốc theo thứ tự xóa
    def clear(i, j):
        if i == j:
            return
        k = pk[(i, j)]
        clear(i + 1, k)
        pairs.append((i, k))
        clear(k + 1, j)
    ans = [""] * (n + 1); pick = [None] * (n + 1)
    for i in range(n - 1, -1, -1):
        if rem[i][n]:
            ans[i], pick[i] = "", n
            continue
        best = None
        for j in range(i, n):
            if rem[i][j] and (best is None or s[j] + ans[j + 1] < best):
                best, pick[i] = s[j] + ans[j + 1], j
        ans[i] = best
    i = 0
    while i < n:
        j = pick[i]
        clear(i, j)
        i = j + 1
    # mô phỏng
    alive = list(range(n))
    for a, b in pairs:
        pa, pb = alive.index(a), alive.index(b)
        assert pb == pa + 1 and is_adj(s[a], s[b])
        del alive[pa:pb + 1]
    return ''.join(s[i] for i in alive), pairs

for s in ["abc", "bcda", "cbadz", "zdce"]:
    res, ops = operations(s)
    assert res == Solution().lexicographicallySmallestString(s)
    print(repr(s), '->', repr(res), '| xóa các cặp (chỉ số gốc):', ops)

'abc' -> 'a' | xóa các cặp (chỉ số gốc): [(1, 2)]
'bcda' -> '' | xóa các cặp (chỉ số gốc): [(1, 2), (0, 3)]
'cbadz' -> 'adz' | xóa các cặp (chỉ số gốc): [(0, 1)]
'zdce' -> 'zdce' | xóa các cặp (chỉ số gốc): []


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
adj(a, b): |a-b| in (1, 25)
rem[i][i] = True; với L chẵn tăng dần:
    rem[i][i+L] = OR_k (adj(s[i], s[k]) and rem[i+1][k] and rem[k+1][i+L]),  k = i+1, i+3, ...
ans[n] = ""
ans[i] = "" nếu rem[i][n], ngược lại min(s[j] + ans[j+1] với rem[i][j])
```

**Checklist:**

- [ ] Cặp `a`–`z` (vòng tròn)?
- [ ] Ghép `s[i]` với **mọi** `s[k]` (lồng nhau), không chỉ ký tự kề?
- [ ] Không tham lam "xóa được thì xóa"?
- [ ] So sánh **cả** chuỗi `s[j] + ans[j+1]`, không chỉ ký tự đầu?
- [ ] `""` là lựa chọn tốt nhất khi xóa sạch được hậu tố?